# NUMIDIA EYE — cloud training ( tabular baseline, experimental )

**Status: EXPERIMENTAL / NOT PRODUCTION.** This notebook reproduces the local `exp-v1` experiment on a cloud machine. It trains nothing new by default — same code (`numidia_ml.cli experiment`), same frozen V2 dataset, same seed. No GPU is needed for this sklearn step (CPU runtime is fine and cheaper); a GPU runtime only matters for the future Sentinel-2 vision experiment.

What this notebook does:
1. Probes the environment (Python + package versions — recorded for reproducibility).
2. Clones the NUMIDIA-EYE repo (pinned commit printed below — record it).
3. Installs the minimal dependencies.
4. Verifies the V2 dataset SHA against the committed manifest (fails loudly on mismatch).
5. Runs the exact local experiment command (train → val-select → test-once).
6. Displays metrics and stages artifacts for download (bring-back, NOT registration).

Bring-back and registration policy: see `docs/cloud-training.md` in the repo. The live API stays `503 AI_UNAVAILABLE` until a model is evaluated, reviewed, and explicitly approved.

In [ ]:
import sys
print('python:', sys.version.split()[0])
for pkg in ('sklearn', 'pandas', 'numpy', 'joblib', 'shapely'):
    try:
        mod = __import__(pkg)
        print(pkg + ':', getattr(mod, '__version__', 'present'))
    except ImportError:
        print(pkg + ': MISSING (installed in the next cell)')


In [ ]:
REPO_URL = 'https://github.com/homixidestayz/NUMIDIA-EYE.git'
REPO_DIR = '/content/NUMIDIA-EYE'
!test -d {REPO_DIR} || git clone --depth 1 {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} rev-parse HEAD
!git -C {REPO_DIR} status --short


In [ ]:
!pip install -q scikit-learn pandas numpy joblib shapely


In [ ]:
import hashlib, json
REPO_DIR = '/content/NUMIDIA-EYE'
manifest = json.load(open(REPO_DIR + '/data/labels/manifest_v2.json'))
want = manifest['dataset_sha256']
h = hashlib.sha256()
with open(REPO_DIR + '/data/labels/firms_labels_v2.csv', 'rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b''):
        h.update(chunk)
got = h.hexdigest()
print('manifest:', want[:12])
print('actual:  ', got[:12])
assert got == want, 'DATASET MISMATCH — stop, do not train on an unverified file'
print('dataset verified:', manifest['dataset_rows'], 'rows')


In [ ]:
!PYTHONPATH=/content/NUMIDIA-EYE/services python -m numidia_ml.cli experiment --dataset /content/NUMIDIA-EYE/data/labels/firms_labels_v2.csv --out /content/exp_cloud --report /content/model-experiment-cloud.md


In [ ]:
import json
m = json.load(open('/content/exp_cloud/metrics.json'))
c = json.load(open('/content/exp_cloud/config.json'))
t = m['test_once']
print('model:', m['model'], '| thr:', t['threshold'], '| seed:', c['random_state'])
print('ROC-AUC: %.4f  PR-AUC: %.4f  acc: %.4f' % (t['roc_auc'], t['pr_auc'], t['accuracy']))
print('P: %.4f  R: %.4f  F1: %.4f  Brier: %.4f  ECE: %.4f' % (t['precision'], t['recall'], t['f1'], t['brier'], t['ece']))
print('confusion:', {k: t[k] for k in ('tp', 'tn', 'fp', 'fn')})
print('config sklearn:', c['sklearn_version'], '| dataset sha:', c['dataset_sha256'][:12])


In [ ]:
!tar -czf /content/numidia_exp_cloud.tar.gz -C /content exp_cloud model-experiment-cloud.md
try:
    from google.colab import files
    files.download('/content/numidia_exp_cloud.tar.gz')
except ImportError:
    print('not on Colab — bundle ready at /content/numidia_exp_cloud.tar.gz (Kaggle: use the output tab)')
